In [1]:
import os, sys, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ["TORCH_NVML_DISABLED"] = "1"
torch.cuda.empty_cache()


os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
import argparse


In [2]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)
ds = VQADataset(config)
df = ds.load_df()


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.language_model.layers.16.mlp.gate_proj.weight'], processor_class=None, tokenizer_class=None, temperature=0.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='fvqa', pred_by='label_maxprob', split='all', suffix=''))


# Get edit_ds by eval

In [3]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)

# model
model = VQAModel(config)

# dataset
ds = VQADataset(config)
random.seed(getattr(config, "seed", 0))
ds.data = random.sample(ds.data, 50)
ds.set_dataloader(shuffle_choices=True)
ds.task_generate(model)
print(ds.task_engineer.eval(ds))

Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.language_model.layers.16.mlp.gate_proj.weight'], processor_class=None, tokenizer_class=None, temperature=0.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='fvqa', pred_by='label_maxprob', split='all', suffix=''))
{'uid': '4546', 'image': 'data/images/fvqa/COCO_val2014_000000014549.jpg', 'question': 'Wha

In [4]:
pred_res_dir = os.path.join("results", "test", "pred", f"{config.model.name}", f"{config.experiment.dataset_name}")
os.makedirs(pred_res_dir, exist_ok=True)
pred_out_path = os.path.join(pred_res_dir, f"{config.experiment.task}_{config.experiment.split}.json")
edit_ds = ds.get_edits()
edit_ds.snap(out_path=pred_out_path)


# edit

In [5]:
# model = VQAModel(config)
# load the prediction set back
import json
pred_set = json.load(open(pred_out_path))
edit_ds = VQADataset(config)
edit_ds.data = pred_set
edit_ds = edit_ds.get_edits()
print(len(edit_ds.data))

2


In [6]:
import copy
model_old = copy.deepcopy(model)
# model_old_weights = copy.deepcopy(model.model.state_dict())
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'sandwich': {'avg_nll': 5.36581563949585,
     'sum_nll': 10.7316312789917,
     'num_tokens': 2,
     'prob': 2.184707292192121e-05},
    'car': {'avg_nll': 3.606623649597168,
     'sum_nll': 3.606623649597168,
     'num_tokens': 1,
     'prob': 0.027148432247252618},
    'tree': {'avg_nll': 0.1066236197948

In [7]:
# minimal single-batch finetune step (ft editor, no history)
editor = get_editor(config, model)
editor.generate = model.model.generate if hasattr(model, 'model') else model.generate
model.model.train()

batch = next(iter(edit_ds.loader))
tokens = model.prepare_training_batch(batch)
editor.edit(config, tokens, batch_history=None)

del tokens
torch.cuda.empty_cache()
# model_new_weights = copy.deepcopy(model.model.state_dict())
model_new = model

Finetuning module model.language_model.layers.16.mlp.gate_proj


In [8]:
edit_ds.task_generate(model_new)
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'boat',
   'label_text': 'boat',
   'label_scores': {'tree': {'avg_nll': 20.125015258789062,
     'sum_nll': 20.125015258789062,
     'num_tokens': 1,
     'prob': 1.8189581148158844e-09},
    'car': {'avg_nll': 13.875014305114746,
     'sum_nll': 13.875014305114746,
     'num_tokens': 1,
     'prob': 9.422445296031577e-07},
    'boat': {'avg_nll': 1.442422035

In [9]:
# model_old = copy.deepcopy(model)
# model_old.model.load_state_dict(model_old_weights)
edit_ds.task_generate(model_old)
edit_ds.data

[{'uid': '4404',
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'tree': {'avg_nll': 0.09182452410459518,
     'sum_nll': 0.09182452410459518,
     'num_tokens': 1,
     'prob': 0.912401326387312},
    'car': {'avg_nll': 3.841824531555176,
     'sum_nll': 3.841824531555176,
     'num_tokens': 1,
     'prob': 0.021457622352790678},
    'boat': {'avg_nll': 2.716824531555176

## eval edits

### reliability

In [10]:
from revlm.metrics import *

In [11]:
reliability(model_old, edit_ds)

0.0

In [12]:
reliability(model_new, edit_ds)

0.5

### generality

In [13]:
edit_uids = [ex["uid"] for ex in edit_ds.data]
edit_uids

['4404', '580']

In [14]:
related_texts = get_t_gen_input("fvqa", edit_ds)
related_texts

{'580': ['Where can the items depicted in this image be located?',
  'In what places can the objects illustrated in this picture be found?',
  'Where are the objects shown in this image typically found?',
  'Can you tell me where the items in this image can be discovered?',
  'Where might one find the objects represented in this picture?',
  'What locations are associated with the items displayed in this image?',
  'Where do the objects featured in this image exist?',
  'In which areas can the items shown in this picture be found?',
  'Where are the objects visible in this image located?',
  'Can you specify where the items in this image can be found?'],
 '4404': ['What is typically present in this location?',
  'What might you expect to discover here?',
  'What is commonly found in this area?',
  'What could you potentially encounter in this place?',
  'What are the usual items or features in this location?',
  'What is likely to be located here?',
  'What can you anticipate finding i

In [15]:

# repo_id = "JJoy333/RationaleVQA"
# local_root = snapshot_download(
#     repo_id=repo_id,
#     repo_type="dataset",
#     allow_patterns=["i_gen/*.parquet"],
# )
# i_gen = pd.read_parquet(os.path.join(local_root, "i_gen", f"{dataset_name}.parquet"))
# i_gen

In [16]:
# def get_i_gen_input(dataset_name: str, edit_ds, k_per_model: int = 2) -> Dict[str, List[str]]:
#     """
#     Build related_images mapping for image_generality by reading existing images only.
#     Does NOT generate new images.
#     """
#     df_full = edit_ds.load_df()
#     image2uid = dict(zip(df_full["image_path"], df_full["uid"].astype(str)))
#     edit_image_paths = {ex["image"] for ex in edit_ds.data}

#     repo_id = "JJoy333/RationaleVQA"
#     local_root = snapshot_download(
#         repo_id=repo_id,
#         repo_type="dataset",
#         allow_patterns=["i_gen/*.parquet"],
#     )
#     i_gen = pd.read_parquet(os.path.join(local_root, "i_gen", f"{dataset_name}.parquet"))
#     i_gen = i_gen[i_gen["image_path"].isin(edit_image_paths)]

#     related_images: Dict[str, List[str]] = {}
#     base_dir = Path("data/related_image") / dataset_name

#     for _, row in i_gen.iterrows():
#         image_path = row["image_path"]
#         uid = image2uid.get(image_path)
#         if uid is None:
#             continue

#         image_info_id = str(row["image_info_id"])
#         img_dir = base_dir / image_info_id
#         if not img_dir.exists():
#             continue

#         img_paths = sorted(str(p) for p in img_dir.glob("*.png"))
#         if not img_paths:
#             continue

#         # optional: apply the k_per_model-per-generator cap

#         related_images.setdefault(uid, []).extend(img_paths)

#     return related_images

In [17]:
related_images = get_i_gen_input("fvqa", edit_ds, k_per_model=2)
related_images

{'581': ['data/related_image/fvqa/val_100132/flux_0.png',
  'data/related_image/fvqa/val_100132/flux_1.png',
  'data/related_image/fvqa/val_100132/flux_2.png',
  'data/related_image/fvqa/val_100132/flux_3.png',
  'data/related_image/fvqa/val_100132/flux_4.png',
  'data/related_image/fvqa/val_100132/sd3_0.png',
  'data/related_image/fvqa/val_100132/sd3_1.png',
  'data/related_image/fvqa/val_100132/sd3_2.png',
  'data/related_image/fvqa/val_100132/sd3_3.png',
  'data/related_image/fvqa/val_100132/sd3_4.png'],
 '4404': ['data/related_image/fvqa/val_105960/flux_0.png',
  'data/related_image/fvqa/val_105960/flux_1.png',
  'data/related_image/fvqa/val_105960/sd3_0.png',
  'data/related_image/fvqa/val_105960/sd3_1.png',
  'data/related_image/fvqa/val_105960/sd3_2.png',
  'data/related_image/fvqa/val_105960/sd3_3.png',
  'data/related_image/fvqa/val_105960/sd3_4.png']}

In [18]:
related_texts

{'580': ['Where can the items depicted in this image be located?',
  'In what places can the objects illustrated in this picture be found?',
  'Where are the objects shown in this image typically found?',
  'Can you tell me where the items in this image can be discovered?',
  'Where might one find the objects represented in this picture?',
  'What locations are associated with the items displayed in this image?',
  'Where do the objects featured in this image exist?',
  'In which areas can the items shown in this picture be found?',
  'Where are the objects visible in this image located?',
  'Can you specify where the items in this image can be found?'],
 '4404': ['What is typically present in this location?',
  'What might you expect to discover here?',
  'What is commonly found in this area?',
  'What could you potentially encounter in this place?',
  'What are the usual items or features in this location?',
  'What is likely to be located here?',
  'What can you anticipate finding i

In [19]:

# related_texts=
# related_images=
# unrelated_texts=
# unrelated_images=

# related_texts={}
# related_images={}
# for ex in edit_ds.data:
#     print(ex)
#     related_texts[ex['uid']] = [ex['question'], ex['question'], ex['question'], ex['question']]
#     related_images[ex['uid']] = [ex['image'], ex['image'], ex['image'], ex['image']]

print(image_generality(model_new, edit_ds, related_images))
print(text_generality(model_new, edit_ds, related_texts))



1.0
0.5


In [20]:
print(locality(model_old, model_new, edit_ds, sample_size=100))


0.97


In [21]:
related_r_gen_df = get_r_gen_input("fvqa")

In [22]:
# 20m30s
editeval(model_old = model_old,
        model_new = model_new,
        edit_ds = edit_ds,
        editor = editor,
        related_texts = related_texts,
        related_images = related_images, 
        related_r_gen_df = related_r_gen_df,
        loc_sample_size = 100)

[Timing] reliability: 0.88s
[Timing] text_generality: 8.64s
[Timing] image_generality: 16.43s
[Timing] rationale_generality: 12.39s


OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 43.19 MiB is free. Including non-PyTorch memory, this process has 79.09 GiB memory in use. Of the allocated memory 77.78 GiB is allocated by PyTorch, and 816.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [23]:
import os

after runing revlm/run/r_gen_image.py

check the completeness of images

In [24]:
r_gen_d = get_r_gen_input("fvqa")
remaining_sid= []
remaining_uid= []
for _, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])



In [25]:
r_gen_d = get_r_gen_input("aokvqa")
remaining_sid= []
remaining_uid= []
remaining_rid= []
for rid, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])
        remaining_rid.append(rid)



Image does not exist: data/r_gen/image/aokvqa/12186_2.png
Image does not exist: data/r_gen/image/aokvqa/12186_3.png
Image does not exist: data/r_gen/image/aokvqa/12186_4.png
Image does not exist: data/r_gen/image/aokvqa/12186_5.png
Image does not exist: data/r_gen/image/aokvqa/12186_6.png
Image does not exist: data/r_gen/image/aokvqa/12187_1.png
Image does not exist: data/r_gen/image/aokvqa/12187_2.png
Image does not exist: data/r_gen/image/aokvqa/12187_3.png
Image does not exist: data/r_gen/image/aokvqa/12187_4.png
Image does not exist: data/r_gen/image/aokvqa/12187_5.png
Image does not exist: data/r_gen/image/aokvqa/12187_6.png
Image does not exist: data/r_gen/image/aokvqa/12188_1.png
Image does not exist: data/r_gen/image/aokvqa/12188_2.png
Image does not exist: data/r_gen/image/aokvqa/12188_3.png
Image does not exist: data/r_gen/image/aokvqa/12188_4.png
Image does not exist: data/r_gen/image/aokvqa/12188_5.png
Image does not exist: data/r_gen/image/aokvqa/12188_6.png
Image does not

In [26]:
print(min(remaining_rid))
print(max(remaining_rid))


88349
89999


In [27]:
# edit_ds.data

# unrelated_ds.df2data(pool_df)


In [29]:
# # Test rationale_generality on a tiny dummy edit set
# import copy
# from revlm.metrics import rationale_generality

# # Require existing model/config/edit_ds from earlier cells
# try:
#     model  # noqa: F401
#     config  # noqa: F401
#     edit_ds  # noqa: F401
# except NameError:
#     raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# # Build a 2-sample dummy edit set from current edit_ds
# _dummy = copy.deepcopy(edit_ds)
# _dummy.data = _dummy.data[:2]
# _dummy.set_dataloader(shuffle_choices=False)

# # Map each uid to the other's uid to form a simple related_rationale
# uids = [_dummy.data[i]["uid"] for i in range(len(_dummy.data))]
# related_rationale = {}
# if len(uids) >= 2:
#     related_rationale = {uids[0]: [uids[1]], uids[1]: [uids[0]]}
# else:
#     # If only one example exists, just point to itself (degenerate case)
#     related_rationale = {uids[0]: [uids[0]]}

# print("rationale_generality:", rationale_generality(model, _dummy, related_rationale))


In [ ]:
# Test edit1_generality on the same tiny dummy edit set
import copy
from revlm.editors import get_editor
from revlm.metrics import edit1_generality, editk_boot_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build dummy edit set (reuse 2 examples)
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
# Keep evaluation deterministic and light
_dummy.config.n_iter = 1  # single training step inside editor.edit
_dummy.set_dataloader(shuffle_choices=True)

# Fresh base model for editing
model_old = copy.deepcopy(model)
editor = get_editor(config, model_old)
editor.generate = model_old.model.generate if hasattr(model_old, 'model') else model_old.generate

print("edit1_generality:", edit1_generality(model_old, _dummy, editor))

print("editk_boot_generality:", editk_boot_generality(model_old, _dummy, editor))


Finetuning module model.language_model.layers.16.mlp.gate_proj
edit1_generality: 0.5
editk_bootstrap_generality: 0.0
